<a href="https://colab.research.google.com/github/atillasahin28/Food-Classification/blob/main/foodposter_nn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!git clone https://github.com/atillasahin28/Food-Classification.git /content/food-classification
%cd /content/food-classification

fatal: destination path '/content/food-classification' already exists and is not an empty directory.
/content/food-classification


In [ ]:
import zipfile, os
os.makedirs("data", exist_ok=True)
ZIP_PATH = "/content/drive/MyDrive/Food-Classification/food-recognition-challenge-2026.zip"
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall("data")
!mv data/train_set/train_set/train_set/* data/train_set/
!rm -rf data/train_set/train_set
!mv data/test_set/test_set/test_set/* data/test_set/
!rm -rf data/test_set/test_set
!echo "Train: $(ls data/train_set/ | wc -l) | Test: $(ls data/test_set/ | wc -l)"

Train: 30612 | Test: 7653


In [ ]:
!mkdir -p results
!cp -r /content/drive/MyDrive/Food-Classification/results/* results/
!echo "Restored:"
!ls results/

Restored:
baseline      combined	   cutmix  geometric  optimized
color_jitter  combined_v2  cutout  mixup


In [ ]:
!pip install grad-cam -q

In [ ]:
%%writefile src/model_v2.py
"""
Deeper ResNet (18-layer style) with wider channels.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)


class FoodResNetV2(nn.Module):
    """
    Deeper ResNet: 4 layers x 3 blocks each = 24 conv layers.
    Wider channels: 64 -> 128 -> 256 -> 512.
    Includes SE (Squeeze-and-Excitation) attention.
    """
    def __init__(self, num_classes=80, dropout=0.4):
        super().__init__()
        self.in_channels = 64

        # Stem: conv -> bn -> relu -> maxpool
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        # 4 layers with 3 blocks each (= 24 conv layers total)
        self.layer1 = self._make_layer(64, 3, stride=1)
        self.layer2 = self._make_layer(128, 3, stride=2)
        self.layer3 = self._make_layer(256, 3, stride=2)
        self.layer4 = self._make_layer(512, 3, stride=2)

        # SE attention on the final features
        self.se_fc1 = nn.Linear(512, 512 // 16)
        self.se_fc2 = nn.Linear(512 // 16, 512)

        # Classifier
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(512, num_classes)
        self._init_weights()

    def _make_layer(self, out_channels, num_blocks, stride):
        layers = [ResidualBlock(self.in_channels, out_channels, stride)]
        self.in_channels = out_channels
        for _ in range(1, num_blocks):
            layers.append(ResidualBlock(out_channels, out_channels, 1))
        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)

        # SE attention
        w = F.relu(self.se_fc1(x))
        w = torch.sigmoid(self.se_fc2(w))
        x = x * w

        x = self.dropout(x)
        x = self.fc(x)
        return x


if __name__ == "__main__":
    model = FoodResNetV2(num_classes=80)
    x = torch.randn(2, 3, 224, 224)
    out = model(x)
    print(f"Input:  {x.shape}")
    print(f"Output: {out.shape}")
    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Overwriting src/model_v2.py


In [ ]:
%%writefile src/train_v2.py
"""
Improved training: deeper model, SGD+momentum, warm restarts, MixUp + geometric.
"""
import os, sys, argparse, json, time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import StratifiedShuffleSplit
from tqdm import tqdm

sys.path.insert(0, os.path.dirname(__file__))
from dataset import FoodDataset, get_transforms
from model_v2 import FoodResNetV2
from mixup import mixup_data, mixup_criterion


class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        image, label = self.subset[idx]
        if self.transform:
            image = self.transform(image)
        return image, label


def train_one_epoch(model, loader, criterion, optimizer, device, use_mixup=True):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, desc="  Train", leave=False):
        images, labels = images.to(device), labels.to(device)

        if use_mixup and np.random.rand() < 0.5:
            # Apply MixUp 50% of the time
            images, labels_a, labels_b, lam = mixup_data(images, labels, alpha=0.3)
            outputs = model(images)
            loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
            _, predicted = outputs.max(1)
            correct += (lam * predicted.eq(labels_a).sum().item()
                        + (1 - lam) * predicted.eq(labels_b).sum().item())
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        total += labels.size(0)

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device, num_classes=80):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    class_correct = np.zeros(num_classes)
    class_total = np.zeros(num_classes)
    all_preds, all_labels = [], []

    for images, labels in tqdm(loader, desc="  Val  ", leave=False):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        _, predicted = outputs.max(1)
        running_loss += loss.item() * images.size(0)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

        for i in range(labels.size(0)):
            label = labels[i].item()
            class_total[label] += 1
            if predicted[i].item() == label:
                class_correct[label] += 1
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    per_class_acc = np.where(class_total > 0, class_correct / class_total, 0.0)
    return (running_loss / total, correct / total, per_class_acc,
            np.array(all_preds), np.array(all_labels))


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--name", type=str, required=True)
    parser.add_argument("--img_size", type=int, default=224)
    parser.add_argument("--batch_size", type=int, default=32)
    parser.add_argument("--epochs", type=int, default=150)
    parser.add_argument("--lr", type=float, default=0.05)
    parser.add_argument("--momentum", type=float, default=0.9)
    parser.add_argument("--weight_decay", type=float, default=5e-4)
    parser.add_argument("--dropout", type=float, default=0.4)
    parser.add_argument("--label_smoothing", type=float, default=0.1)
    parser.add_argument("--num_workers", type=int, default=2)
    parser.add_argument("--data_dir", type=str, default="data")
    parser.add_argument("--results_dir", type=str, default="results")
    args = parser.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    save_dir = os.path.join(args.results_dir, args.name)
    os.makedirs(save_dir, exist_ok=True)
    with open(os.path.join(save_dir, "config.json"), "w") as f:
        json.dump(vars(args), f, indent=2)

    # Use combined augmentation (geometric + color + cutout)
    train_transform, val_transform = get_transforms("combined", args.img_size)

    full_dataset = FoodDataset(
        csv_file=os.path.join(args.data_dir, "train_labels.csv"),
        img_dir=os.path.join(args.data_dir, "train_set"),
        transform=None,
    )

    labels_array = full_dataset.df["label"].values
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
    train_idx, val_idx = next(splitter.split(np.zeros(len(labels_array)), labels_array))

    train_data = TransformSubset(Subset(full_dataset, train_idx), train_transform)
    val_data = TransformSubset(Subset(full_dataset, val_idx), val_transform)

    train_loader = DataLoader(train_data, batch_size=args.batch_size, shuffle=True,
                              num_workers=args.num_workers, pin_memory=True)
    val_loader = DataLoader(val_data, batch_size=args.batch_size, shuffle=False,
                            num_workers=args.num_workers, pin_memory=True)

    print(f"Train: {len(train_data)} | Val: {len(val_data)}")

    # Deeper model
    model = FoodResNetV2(num_classes=80, dropout=args.dropout).to(device)
    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

    # SGD with momentum
    criterion = nn.CrossEntropyLoss(label_smoothing=args.label_smoothing)
    optimizer = torch.optim.SGD(model.parameters(), lr=args.lr,
                                 momentum=args.momentum, weight_decay=args.weight_decay)

    # Cosine annealing with warm restarts (restart every 50 epochs)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=50, T_mult=1, eta_min=1e-5)

    best_val_acc = 0.0
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": []}

    for epoch in range(1, args.epochs + 1):
        print(f"\nEpoch {epoch}/{args.epochs}")
        start = time.time()

        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, device, use_mixup=True)
        val_loss, val_acc, per_class_acc, preds, labels = evaluate(
            model, val_loader, criterion, device)
        scheduler.step()

        elapsed = time.time() - start
        lr = optimizer.param_groups[0]["lr"]

        print(f"  Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f}")
        print(f"  Val   Loss: {val_loss:.4f}  Acc: {val_acc:.4f}  LR: {lr:.5f}  ({elapsed:.1f}s)")

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["lr"].append(lr)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), os.path.join(save_dir, "best_model.pth"))
            np.save(os.path.join(save_dir, "best_per_class_acc.npy"), per_class_acc)
            np.save(os.path.join(save_dir, "best_preds.npy"), preds)
            np.save(os.path.join(save_dir, "best_labels.npy"), labels)
            print(f"  >>> New best: {val_acc:.4f}")

    with open(os.path.join(save_dir, "history.json"), "w") as f:
        json.dump(history, f)
    print(f"\nDone! Best val accuracy: {best_val_acc:.4f}")


if __name__ == "__main__":
    main()

Overwriting src/train_v2.py


In [ ]:
%%writefile src/predict_v2_tta.py
import os, sys, argparse, torch, pandas as pd, numpy as np
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import torchvision.transforms as T
from tqdm import tqdm

sys.path.insert(0, os.path.dirname(__file__))
from model_v2 import FoodResNetV2

class TTATestDataset(Dataset):
    def __init__(self, img_dir, transform_list):
        self.img_dir = img_dir
        self.img_names = sorted(os.listdir(img_dir))
        self.transform_list = transform_list
    def __len__(self):
        return len(self.img_names)
    def __getitem__(self, idx):
        img_name = self.img_names[idx]
        image = Image.open(os.path.join(self.img_dir, img_name)).convert("RGB")
        augmented = [t(image) for t in self.transform_list]
        return torch.stack(augmented), img_name

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--experiment", type=str, default="optimized")
    parser.add_argument("--data_dir", type=str, default="data")
    parser.add_argument("--results_dir", type=str, default="results")
    parser.add_argument("--img_size", type=int, default=224)
    parser.add_argument("--batch_size", type=int, default=16)
    args = parser.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = FoodResNetV2(num_classes=80)
    model.load_state_dict(torch.load(
        os.path.join(args.results_dir, args.experiment, "best_model.pth"),
        map_location=device))
    model.eval().to(device)

    normalize = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    sz = args.img_size

    tta_transforms = [
        T.Compose([T.Resize((sz, sz)), T.ToTensor(), normalize]),
        T.Compose([T.Resize((sz, sz)), T.RandomHorizontalFlip(p=1.0), T.ToTensor(), normalize]),
        T.Compose([T.Resize((int(sz*1.15), int(sz*1.15))), T.CenterCrop(sz), T.ToTensor(), normalize]),
        T.Compose([T.Resize((int(sz*1.15), int(sz*1.15))), T.FiveCrop(sz), lambda c: c[0], T.ToTensor(), normalize]),
        T.Compose([T.Resize((int(sz*1.15), int(sz*1.15))), T.FiveCrop(sz), lambda c: c[1], T.ToTensor(), normalize]),
        T.Compose([T.Resize((int(sz*1.15), int(sz*1.15))), T.FiveCrop(sz), lambda c: c[2], T.ToTensor(), normalize]),
        T.Compose([T.Resize((int(sz*1.15), int(sz*1.15))), T.FiveCrop(sz), lambda c: c[3], T.ToTensor(), normalize]),
        T.Compose([T.Resize((int(sz*1.2), int(sz*1.2))), T.RandomCrop(sz), T.ToTensor(), normalize]),
        T.Compose([T.Resize((int(sz*1.15), int(sz*1.15))), T.CenterCrop(sz), T.RandomHorizontalFlip(p=1.0), T.ToTensor(), normalize]),
        T.Compose([T.Resize((sz, sz)), T.ColorJitter(brightness=0.1, contrast=0.1), T.ToTensor(), normalize]),
    ]

    dataset = TTATestDataset(os.path.join(args.data_dir, "test_set"), tta_transforms)
    loader = DataLoader(dataset, batch_size=args.batch_size, shuffle=False, num_workers=2)

    all_names, all_preds = [], []
    with torch.no_grad():
        for augmented_batch, img_names in tqdm(loader, desc="TTA Predicting"):
            B, N, C, H, W = augmented_batch.shape
            images = augmented_batch.view(B * N, C, H, W).to(device)
            outputs = model(images)
            probs = F.softmax(outputs, dim=1)
            probs = probs.view(B, N, -1).mean(dim=1)
            preds = probs.argmax(dim=1).cpu().numpy() + 1
            all_names.extend(img_names)
            all_preds.extend(preds.tolist())

    submission = pd.DataFrame({"img_name": all_names, "label": all_preds})
    os.makedirs("submissions", exist_ok=True)
    out_path = os.path.join("submissions", f"submission_{args.experiment}_tta.csv")
    submission.to_csv(out_path, index=False)
    print(f"Saved {out_path} ({len(submission)} predictions)")

if __name__ == "__main__":
    main()

Overwriting src/predict_v2_tta.py


In [ ]:
!cd /content/food-classification && python src/model_v2.py

Input:  torch.Size([2, 3, 224, 224])
Output: torch.Size([2, 80])
Parameters: 17,521,584


In [ ]:
!python src/train_v2.py --name optimized --epochs 150 --batch_size 64 --img_size 224 --num_workers 4

Device: cuda
Train: 26020 | Val: 4592
Parameters: 17,521,584

Epoch 1/150
  Train Loss: 4.3047  Acc: 0.0320
  Val   Loss: 4.1602  Acc: 0.0477  LR: 0.04995  (70.2s)
  >>> New best: 0.0477

Epoch 2/150
  Train Loss: 4.1575  Acc: 0.0551
  Val   Loss: 4.2659  Acc: 0.0475  LR: 0.04980  (69.9s)

Epoch 3/150
  Train Loss: 4.0687  Acc: 0.0702
  Val   Loss: 3.9012  Acc: 0.0995  LR: 0.04956  (69.3s)
  >>> New best: 0.0995

Epoch 4/150
  Train Loss: 4.0010  Acc: 0.0840
  Val   Loss: 3.9563  Acc: 0.0943  LR: 0.04921  (69.1s)

Epoch 5/150
  Train Loss: 3.9335  Acc: 0.0997
  Val   Loss: 4.0149  Acc: 0.0880  LR: 0.04878  (69.2s)

Epoch 6/150
  Train Loss: 3.8867  Acc: 0.1082
  Val   Loss: 4.0268  Acc: 0.0847  LR: 0.04824  (69.1s)

Epoch 7/150
  Train Loss: 3.8094  Acc: 0.1258
  Val   Loss: 3.7070  Acc: 0.1485  LR: 0.04762  (69.0s)
  >>> New best: 0.1485

Epoch 8/150
  Train Loss: 3.7511  Acc: 0.1428
  Val   Loss: 3.6935  Acc: 0.1453  LR: 0.04691  (69.1s)

Epoch 9/150
  Train Loss: 3.6866  Acc: 0.1601

In [ ]:
import shutil, os
DRIVE_SAVE = "/content/drive/MyDrive/Food-Classification/results"
os.makedirs(DRIVE_SAVE, exist_ok=True)
shutil.copytree("results", DRIVE_SAVE, dirs_exist_ok=True)
print("Saved!")

In [ ]:
!python src/predict_v2_tta.py --experiment optimized --img_size 224

TTA Predicting: 100% 479/479 [01:44<00:00,  4.58it/s]
Saved submissions/submission_optimized_tta.csv (7653 predictions)


In [ ]:
import shutil
shutil.copytree("submissions", "/content/drive/MyDrive/Food-Classification/submissions", dirs_exist_ok=True)

from google.colab import files
files.download("submissions/submission_optimized_tta.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>